# Seguridad ciudadana en el Perú mediante Machine Learning

Objetivo: reproducir validación, integración, EDA, clustering, clasificación, XAI e interpretación ética usando datos oficiales.

## Fuentes y metodología

Las fuentes están en `data/raw`. El nivel común validado es departamento-año para 2018-2024. Antes de modelar se inspeccionan columnas, nulos, duplicados y granularidad.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import numpy as np
SEED = 42
ROOT

WindowsPath('C:/Users/Iriarte 06/OneDrive/Documentos/ChatGPT/Parcial Criminalidad/criminalidad-peru-ml')

## Validación e integración

Se ejecuta el pipeline de procesamiento para construir el panel reproducible.

In [2]:
from data_processing import build_department_year_dataset
panel = build_department_year_dataset()
panel.shape, panel.head()

((182, 38),
   departamento  anio  denuncias_total  meses_observados  anio_completo  \
 0     AMAZONAS  2018             9193                12           True   
 1     AMAZONAS  2019             9927                12           True   
 2     AMAZONAS  2020             7917                12           True   
 3     AMAZONAS  2021             8679                12           True   
 4     AMAZONAS  2022             9057                12           True   
 
    denuncias_estafa  denuncias_extorsion  denuncias_hurto  denuncias_otros  \
 0             116.0                 23.0           1903.0           4238.0   
 1             124.0                 13.0           2090.0           4149.0   
 2             123.0                 16.0           1584.0           3438.0   
 3             233.0                 31.0           1831.0           3618.0   
 4             312.0                 54.0           1858.0           3824.0   
 
    denuncias_robo  ...  victimizacion_estafa_pct  victimiza

## EDA

Se generan tablas y figuras descriptivas. Cada figura se guarda en `outputs/figures`.

In [3]:
from modeling import setup, run_eda, run_clustering, run_classification
setup()
eda = run_eda(panel)
pd.DataFrame(eda['by_year'])

,anio,denuncias_total,denuncias_tasa_100k,victimizacion_pct,percepcion_inseguridad_pct,confianza_pnp_pct
0,2018,774804,2142.362270,23.711538,50.657692,16.646154
1,2019,854898,2448.644999,23.888462,50.246154,18.534615
2,2020,649169,1959.945626,20.373077,25.046154,26.773077
3,2021,754638,2280.238181,15.584615,47.157692,22.034615
4,2022,877736,2506.311656,20.219231,54.534615,19.869231
5,2023,1044843,2773.142250,23.046154,51.588462,19.215385
6,2024,1014315,2683.897991,24.457692,51.050000,17.730769


Interpretación: revise la evolución anual para distinguir denuncias registradas, victimización, percepción y confianza. Estos conceptos no son equivalentes.

## Clustering

Se comparan K-Means y Agglomerative Clustering con variables estandarizadas.

In [4]:
clustering = run_clustering(panel)
clustering['best_model'], clustering['k']

('KMeans', 3)

In [5]:
pd.read_csv(ROOT / 'outputs' / 'tables' / 'cluster_profiles.csv')

,cluster,observaciones,territorios,denuncias_tasa_100k,victimization,percepcion,confianza,brecha
0,0,41,"APURIMAC, AREQUIPA, AYACUCHO, CUSCO, HUANCAVEL...",2509.674320,26.836585,54.002439,13.236585,27.165854
1,1,61,"AMAZONAS, ANCASH, AREQUIPA, AYACUCHO, CAJAMARC...",1877.525431,17.514754,31.601639,23.637705,14.086885
2,2,59,"AREQUIPA, AYACUCHO, CAJAMARCA, HUANUCO, ICA, J...",2861.843899,21.045763,56.001695,20.120339,34.955932


Interpretación: los clusters son perfiles exploratorios de patrones agregados; no son etiquetas de peligrosidad.

## Clasificación

Se predice victimización alta relativa en t usando variables históricas y rezagos. Se evita leakage temporal.

In [6]:
classification = run_classification(panel)
pd.read_csv(ROOT / 'outputs' / 'tables' / 'classification_benchmark.csv')

,modelo,accuracy,precision,recall,f1,roc_auc,pr_auc,cv_best_score
0,DummyClassifier,0.760870,0.000000,0.0,0.000000,0.500000,0.239130,NaN
1,LogisticRegression,0.521739,0.333333,1.0,0.500000,0.974026,0.908799,0.687179
2,RandomForest,0.891304,0.687500,1.0,0.814815,1.000000,1.000000,0.671429


Interpretación: compare contra DummyClassifier. Accuracy no basta si el baseline no recupera la clase positiva.

## XAI, ética y conclusiones

La importancia por permutación indica relevancia predictiva, no causalidad. Deben considerarse sesgos de reporte, cobertura, medición y encuesta, además del riesgo de estigmatización territorial.

In [7]:
pd.read_csv(ROOT / 'outputs' / 'tables' / 'classification_permutation_importance.csv').head(10)

,feature,importance_mean,importance_std
0,denuncias_tasa_100k_lag1,0.031985,0.021878
1,percepcion_inseguridad_pct_lag1,0.031706,0.026928
2,diversidad_modalidades,0.021825,0.012601
3,denuncias_total_lag1,0.018188,0.014088
4,denuncias_tasa_yoy,0.007275,0.012601
5,prop_estafa,0.003638,0.009624
6,prop_robo,0.000000,0.000000
7,prop_hurto,0.000000,0.000000
8,prop_violencia_contra_la_mujer_e_integrantes,0.000000,0.000000
9,prop_extorsion,0.000000,0.000000
